## simple Gen AI App using Langchain

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = os.getenv('LANGCHAIN_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv('LANGCHAIN_PROJECT')

In [9]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
loader 

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [10]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialPolly AI assistantTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run previe

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTutorialPolly AI assistantTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run previe

In [12]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [13]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)
vectorstoredb

In [14]:
query = "Once your app is working well in prototyping, you release it to a small group"
result = vectorstoredb.similarity_search(query)
result[0].page_content

'automation rulesConfigure webhook notifications for rulesFeedback & evaluationManage evaluatorsLog user feedback using the SDKCollect feedback with presigned URLsSet up online evaluatorsMonitoring & alertingMonitor projects with dashboardsAlertsInsightsData type referenceRun (span) data formatFeedback data formatTrace query syntaxOn this pagePrerequisitesPrototypingSet up your environmentTrace LLM callsTrace the whole pipelineCheck your traces from the terminalBeta testingCollect feedbackLog metadataProductionMonitoringA/B testingDrilldownConclusionTrace an LLM application tutorialCopy pageAdd LangSmith observability to an LLM application across prototyping, beta testing, and production.Copy pageIn this tutorial, you will build a customer support chatbot using retrieval-augmented generation (RAG) and add LangSmith observability at each stage of development, from early prototyping through production.'

In [16]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")
print(llm) 

output_version=None profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000021C83E23D30> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021C83D5D5D0> root_client=<openai.OpenAI object at 0x0000021C83E23F70> root_async_client=<openai.AsyncOpenAI object at 0x0000021C83E23DF0> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('***

In [ ]:
## here we define the retriever, which is a wrapper around the vectorstoredb that allows us to easily retrieve relevant documents based on a query. We specify that we want to retrieve the top 3 most relevant documents for each query.

retrieval = vectorstoredb.as_retriever(search_kwargs={"k": 3})

In [ ]:
## here we use the retriever to retrieve the relevant documents for the query "What is LangSmith?" and print the number of documents retrieved and the first 200 characters of the content of the first document.
docs = retrieval.invoke("What is LangSmith?")
print(len(docs))
print(docs[0].page_content[:200])

3
​Conclusion
In this tutorial, you added LangSmith observability to an application across its full development lifecycle. The same tracing setup that helps you iterate quickly during prototyping will c


In [24]:
docs = retrieval.invoke("What is LangSmith?")

total_chars = sum(len(d.page_content) for d in docs)
print("Total characters:", total_chars)

for i, d in enumerate(docs):
    print(f"Doc {i} length:", len(d.page_content))

Total characters: 2742
Doc 0 length: 902
Doc 1 length: 991
Doc 2 length: 849


In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question using the provided context.
If you don't know, say you don't know.

<context>
{context}
</context>
"""),
    ("human", "{input}")
])

document_chain = create_stuff_documents_chain(llm, prompt)

In [19]:
from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retrieval, document_chain)

In [25]:
response = retrieval_chain.invoke({
    "input": "What is LangSmith?"
})

print(response["answer"])

LangSmith is a tool that provides observability for applications across their development lifecycle. It offers tracing setup to help developers iterate quickly during prototyping and continue to provide visibility into individual traces and aggregate performance trends in production. It integrates with various providers like LangChain, LangGraph, and Anthropic, and offers features like monitoring, logging metadata, managing traces, and setting up automation rules.
